# Étape 5a - Enrichissement : jointures vérifiées

Ajoute aux ventes Silver la date, le produit et le magasin, puis renomme
les colonnes et crée des colonnes calculées.

- Entrée : `data/silver/{sales, dates, products, stores}`.
- Sortie : `data/silver/sales_enriched/` (Parquet). La granularité reste : une ligne = un produit dans un ticket.

Les relations ont été vérifiées au Jour 1 (`resultats_relations.csv`) et à
l'étape 2 : chaque clé de vente existe dans sa table de référence.

**Organisation du notebook**

| Partie | Rôle |
|---|---|
| 0. Paramètres | Chemins et session Spark lus dans `config/parametres.json` ; journal |
| Lecture, `joindre` | **Lecture** des 4 jeux Silver et **fonction** de jointure contrôlée |
| 1 à 4 | **Transformations** : 3 jointures N:1, renommage, colonnes calculées |
| 5 | **Écriture** de `silver/sales_enriched` (partitionné par `annee`), **contrôle** du nombre de lignes et **résultats** : `outputs/bilan_jointures.json` |

## 0. Paramètres, chemins, session Spark et journal

Tous les paramètres et chemins viennent de `config/parametres.json` : aucun chemin n'est écrit en dur dans ce notebook.
Le journal d'exécution est ajouté dans `outputs/logs/<notebook>.log`.

In [1]:
import json
import os
from datetime import datetime

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

# ---------- Parametres et chemins (config/parametres.json) ----------
RACINE = os.getcwd()
while not os.path.exists(os.path.join(RACINE, "config", "parametres.json")):
    if os.path.dirname(RACINE) == RACINE:
        raise RuntimeError("config/parametres.json introuvable : lancer le notebook depuis le projet")
    RACINE = os.path.dirname(RACINE)
with open(os.path.join(RACINE, "config", "parametres.json")) as f:
    PARAMS = json.load(f)


def chemin(cle):
    """Chemin absolu d'une entree de PARAMS["chemins"] (relative a la racine, ~ accepte)."""
    return os.path.normpath(os.path.join(RACINE, os.path.expanduser(PARAMS["chemins"][cle])))


LOCAL = "file://"  # sans ce prefixe, Spark irait chercher les fichiers dans HDFS (fs.defaultFS)
SOURCE = chemin("source_tpcxbb")
SILVER = LOCAL + chemin("silver")
GOLD = LOCAL + chemin("gold")
OUTPUTS = chemin("outputs")
DOSSIER_SCHEMAS = chemin("schemas")
# store_sales : echantillon ou fichier complet selon le parametre "echantillon.actif"
FICHIER_VENTES = (chemin("echantillon_ventes") if PARAMS["echantillon"]["actif"]
                  else os.path.join(SOURCE, "store_sales", "store_sales.dat"))
os.makedirs(chemin("logs"), exist_ok=True)

# ---------- Journal d'execution ----------
FICHIER_JOURNAL = os.path.join(chemin("logs"), "enrichissement_ventes.log")


def journal(*message):
    ligne = datetime.now().strftime("%Y-%m-%d %H:%M:%S") + " | enrichissement_ventes | " + " ".join(str(m) for m in message)
    print(ligne)
    with open(FICHIER_JOURNAL, "a") as f:
        f.write(ligne + "\n")


# ---------- Session Spark (parametres "spark") ----------
constructeur = SparkSession.builder.appName("TP3-Enrichissement").master(PARAMS["spark"]["master"])
for cle, valeur in PARAMS["spark"]["config"].items():
    constructeur = constructeur.config(cle, valeur)
spark = constructeur.getOrCreate()
spark.sparkContext.setLogLevel(PARAMS["spark"]["niveau_journal"])

journal("debut", "| echantillon" if PARAMS["echantillon"]["actif"] else "| fichier complet",
        "| Spark", spark.version, PARAMS["spark"]["master"])

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 17:29:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-11-18 17:29:38 | enrichissement_ventes | debut | echantillon | Spark 4.3.0 local[2]


In [2]:
ventes = spark.read.parquet(SILVER + "/sales")
dates = spark.read.parquet(SILVER + "/dates")
produits = spark.read.parquet(SILVER + "/products")
magasins = spark.read.parquet(SILVER + "/stores")

jointures = []  # documentation de chaque jointure


def joindre(gauche, droite, cle_gauche, cle_droite, nom):
    """Jointure gauche N:1 controlee : lignes avant/apres et lignes non rapprochees."""
    # Cardinalite attendue cote reference : la cle doit etre unique
    cle_unique = droite.count() == droite.select(cle_droite).distinct().count()
    avant = gauche.count()
    resultat = gauche.join(droite, gauche[cle_gauche] == droite[cle_droite], "left")
    apres = resultat.count()
    non_rapproches = resultat.filter(droite[cle_droite].isNull()).count()
    info = {"jointure": nom, "colonnes": f"{cle_gauche} = {cle_droite}", "relation": "N:1 (plusieurs ventes -> une reference)",
            "cle_reference_unique": cle_unique, "lignes_avant": avant, "lignes_apres": apres,
            "non_rapprochees": non_rapproches}
    jointures.append(info)
    print(json.dumps(info, indent=1, ensure_ascii=False))
    assert cle_unique and apres == avant, "La jointure a duplique des lignes : a analyser"
    return resultat.drop(droite[cle_droite])

## 1. Ventes x dates

- Sources : `store_sales` et `date_dim`.
- Colonnes : `ss_sold_date_sk = d_date_sk`.
- Relation N:1 : chaque vente a une date, et une date regroupe plusieurs ventes.
- Cardinalité attendue : autant de lignes avant qu'après, 0 vente non rapprochée.

In [3]:
v = joindre(ventes, dates.select("d_date_sk", "d_date", "d_year", "d_qoy", "d_moy", "d_day_name", "d_weekend"),
            "ss_sold_date_sk", "d_date_sk", "store_sales -> date_dim")

{
 "jointure": "store_sales -> date_dim",
 "colonnes": "ss_sold_date_sk = d_date_sk",
 "relation": "N:1 (plusieurs ventes -> une reference)",
 "cle_reference_unique": true,
 "lignes_avant": 66593,
 "lignes_apres": 66593,
 "non_rapprochees": 0
}


## 2. Ventes x produits

- Sources : `store_sales` et `item`.
- Colonnes : `ss_item_sk = i_item_sk`.
- Relation N:1 (un produit est vendu dans plusieurs lignes).

In [4]:
v = joindre(v, produits.select("i_item_sk", "i_item_id", "i_product_name", "i_category_id", "i_category",
                               "i_class", "i_brand"),
            "ss_item_sk", "i_item_sk", "store_sales -> item")

{
 "jointure": "store_sales -> item",
 "colonnes": "ss_item_sk = i_item_sk",
 "relation": "N:1 (plusieurs ventes -> une reference)",
 "cle_reference_unique": true,
 "lignes_avant": 66593,
 "lignes_apres": 66593,
 "non_rapprochees": 0
}


## 3. Ventes x magasins

- Sources : `store_sales` et `store`.
- Colonnes : `ss_store_sk = s_store_sk`.
- Relation N:1 (12 magasins pour toutes les lignes de vente).

In [5]:
v = joindre(v, magasins.select("s_store_sk", "s_store_id", "s_store_name", "s_city", "s_state"),
            "ss_store_sk", "s_store_sk", "store_sales -> store")

{
 "jointure": "store_sales -> store",
 "colonnes": "ss_store_sk = s_store_sk",
 "relation": "N:1 (plusieurs ventes -> une reference)",
 "cle_reference_unique": true,
 "lignes_avant": 66593,
 "lignes_apres": 66593,
 "non_rapprochees": 0
}


## 4. Renommage, colonnes calculées et sélection

Noms explicites en français. Colonnes calculées :
- `cout_total` = `ss_ext_wholesale_cost` ;
- `taux_marge` = marge ÷ montant net payé (vide si le montant payé est 0) ;
- `taux_remise` = remise ÷ prix catalogue total (vide si le prix catalogue est 0).

In [6]:
ventes_enrichies = v.select(
    # Quand ?
    F.col("d_date").alias("date_vente"),
    F.col("d_year").alias("annee"),
    F.col("d_qoy").alias("trimestre"),
    F.col("d_moy").alias("mois"),
    F.col("d_day_name").alias("jour_semaine"),
    F.col("ss_sold_time_sk").alias("heure_sk"),
    # Quel achat ?
    F.col("ss_ticket_number").alias("ticket_id"),
    F.col("ss_ligne_ticket").alias("ligne_ticket"),
    F.col("ss_customer_sk").alias("client_sk"),
    F.col("ss_promo_sk").alias("promotion_sk"),
    # Quel produit ?
    F.col("ss_item_sk").alias("produit_sk"),
    F.col("i_item_id").alias("produit_id"),
    F.col("i_product_name").alias("nom_produit"),
    F.col("i_category_id").alias("categorie_id"),
    F.col("i_category").alias("categorie"),
    F.col("i_class").alias("sous_categorie"),
    F.col("i_brand").alias("marque"),
    # Quel magasin ?
    F.col("ss_store_sk").alias("magasin_sk"),
    F.col("s_store_id").alias("magasin_id"),
    F.col("s_store_name").alias("nom_magasin"),
    F.col("s_city").alias("ville"),
    F.col("s_state").alias("etat"),
    # Mesures
    F.col("ss_quantity").alias("quantite"),
    F.col("ss_list_price").alias("prix_catalogue_unitaire"),
    F.col("ss_sales_price").alias("prix_vente_unitaire"),
    F.col("ss_ext_list_price").alias("montant_catalogue"),
    F.col("ss_ext_discount_amt").alias("montant_remise"),
    F.col("ss_ext_sales_price").alias("montant_vente"),
    F.col("ss_coupon_amt").alias("montant_coupon"),
    F.col("ss_net_paid").alias("montant_net_paye"),
    F.col("ss_ext_wholesale_cost").alias("cout_total"),
    F.col("ss_net_profit").alias("marge"),
    # Indicateurs de qualite (etape 4)
    F.col("ss_prix_nul").alias("prix_nul"),
    F.col("ss_marge_negative").alias("marge_negative"),
    F.col("ss_sans_promotion").alias("sans_promotion"),
    # Provenance : fichier brut d'origine de la vente
    F.col("_source_systeme"),
    F.col("_source_fichier"),
).withColumn(
    "_sources_silver", F.lit("silver/sales, silver/dates, silver/products, silver/stores")
).withColumn(
    "_traitement", F.lit("notebooks/transformations/enrichissement_ventes.ipynb")
).withColumn(
    "_date_traitement", F.current_timestamp()
).withColumn(
    "taux_marge", F.when(F.col("montant_net_paye") != 0,
                         F.round(F.col("marge") / F.col("montant_net_paye"), 4))
).withColumn(
    "taux_remise", F.when(F.col("montant_catalogue") != 0,
                          F.round(F.col("montant_remise") / F.col("montant_catalogue"), 4))
)

ventes_enrichies.printSchema()
ventes_enrichies.select("date_vente", "ticket_id", "nom_produit", "categorie", "nom_magasin", "quantite",
                        "montant_net_paye", "marge", "taux_remise").show(10, truncate=False)

root
 |-- date_vente: date (nullable = true)
 |-- annee: integer (nullable = true)
 |-- trimestre: integer (nullable = true)
 |-- mois: integer (nullable = true)
 |-- jour_semaine: string (nullable = true)
 |-- heure_sk: long (nullable = true)
 |-- ticket_id: long (nullable = true)
 |-- ligne_ticket: integer (nullable = true)
 |-- client_sk: long (nullable = true)
 |-- promotion_sk: long (nullable = true)
 |-- produit_sk: long (nullable = true)
 |-- produit_id: string (nullable = true)
 |-- nom_produit: string (nullable = true)
 |-- categorie_id: integer (nullable = true)
 |-- categorie: string (nullable = true)
 |-- sous_categorie: string (nullable = true)
 |-- marque: string (nullable = true)
 |-- magasin_sk: long (nullable = true)
 |-- magasin_id: string (nullable = true)
 |-- nom_magasin: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- etat: string (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- prix_catalogue_unitaire: decimal(7,2) (nullable = tr

+----------+---------+---------------------------------------------+----------------------+-----------+--------+----------------+--------+-----------+
|date_vente|ticket_id|nom_produit                                  |categorie             |nom_magasin|quantite|montant_net_paye|marge   |taux_remise|
+----------+---------+---------------------------------------------+----------------------+-----------+--------+----------------+--------+-----------+
|2001-01-01|0        |vwPR6l3UPyCIv5bljZbIG9k9YHLajogyqKbd6        |Home & Kitchen        |Avalon     |54      |4500.36         |-856.98 |0.4400     |
|2001-01-01|0        |6VxbaI6thqY49zO                              |Music                 |Avalon     |31      |715.48          |157.48  |0.2502     |
|2001-01-01|20       |p0d9bO6SHRcaaGUro02zVXid0KUzssrVX            |Music                 |Buffalo    |90      |384.30          |-5262.30|0.9600     |
|2001-01-01|30       |OZEp2D0icEeiNK1wTTOsPcCrGg0XG7y              |Music                 |Elm

## 5. Filtrage selon une règle explicite

Aucune ligne n'est retirée ici : le filtrage des lignes invalides a été fait à
l'étape 4 (rejets). Les lignes à prix nul ou à marge négative sont gardées, mais
signalées, pour que chaque indicateur Gold puisse décider de les inclure ou non.

In [7]:
assert ventes_enrichies.count() == ventes.count()
ventes_enrichies.write.mode("overwrite").partitionBy("annee").parquet(SILVER + "/sales_enriched")

relu = spark.read.parquet(SILVER + "/sales_enriched")
with open(os.path.join(DOSSIER_SCHEMAS, "silver_sales_enriched.json"), "w") as f:
    json.dump(json.loads(relu.schema.json()), f, indent=1)
print("Lignes ecrites :", relu.count())
relu.groupBy("annee").count().orderBy("annee").show()

with open(os.path.join(OUTPUTS, "bilan_jointures.json"), "w") as f:
    json.dump(jointures, f, indent=1, ensure_ascii=False)

Lignes ecrites : 66593


+-----+-----+
|annee|count|
+-----+-----+
| 2001|13381|
| 2002|13404|
| 2003|13463|
| 2004|13590|
| 2005|12755|
+-----+-----+



In [8]:
journal("fin | OK")
spark.stop()

2026-11-18 17:30:05 | enrichissement_ventes | fin | OK
